# R4 · Warm-up: the first logistic regression and the threshold

**Type**: Implement with scikit-learn · **Level**: ●○○ guided · **Time**: about 25 minutes · **core path ★**

## Objective

Train a logistic regression in a `Pipeline` with scaling, read accuracy and AUC, and see what happens to **precision and recall** when the decision threshold is moved away from 0.5.

## Dataset

`load_breast_cancer`: 569 tumors with 30 measurements; `y = 1` if the tumor is **malignant** (37%). Stratified 75/25 split.

## Educational purpose

Understand that `predict` uses the 0.5 threshold out of habit and not out of necessity: when missing a malignant case costs more than a false alarm, you lower the threshold and accept losing precision to gain recall.

## Task

1. Build `pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))`, train it and save `test_acc` (accuracy) and `test_auc`.
2. `p_mal = pipe.predict_proba(X_te)[:, 1]`. For the thresholds `[0.5, 0.2, 0.05]` build `table` (index: threshold, columns `recall` and `precision` of the malignant class).
3. Save `screening_threshold`: the **highest** threshold of the grid `np.linspace(0.01, 0.99, 99)` for which the recall of the malignant cases is at least 0.98.
4. Without scaler: `acc_without_scaler` and `iter_without` (`n_iter_[0]` of a `LogisticRegression(max_iter=1000)` trained directly on `X_tr`).

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

data = load_breast_cancer(as_frame=True)
X, y = data.data, (data.target == 0).astype(int)                       # 1 = malignant
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
print(X_tr.shape, X_te.shape, "| share of malignant cases:", round(float(y.mean()), 3))

## Your work

In [ ]:
pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(..., ...)
test_acc = accuracy_score(y_te, pipe.predict(X_te))
test_auc = roc_auc_score(y_te, pipe.predict_proba(X_te)[:, ...])

In [ ]:
p_mal = pipe.predict_proba(X_te)[:, 1]
table = pd.DataFrame(index=[0.5, 0.2, 0.05], columns=["recall", "precision"], dtype=float)
for threshold in table.index:
    predicted = (p_mal >= threshold).astype(int)
    table.loc[threshold] = [recall_score(y_te, predicted), precision_score(y_te, ...)]
table.round(3)

In [ ]:
thresholds = np.linspace(0.01, 0.99, 99)
recalls = np.array([recall_score(y_te, (p_mal >= s).astype(int)) for s in thresholds])
screening_threshold = ...

In [ ]:
without = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
acc_without_scaler = ...
iter_without = ...

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`thresholds[recalls >= 0.98].max()`. The recall of a class: `recall_score(y_te, predicted)` with `predicted = (p_mal >= threshold).astype(int)`.'):
    check(0.93 < test_acc < 0.99 and test_auc > 0.98, "accuracy between 0.93 and 0.99, AUC over 0.98", "auc = roc_auc_score(y_te, p_mal)")
    check(list(table.index) == [0.5, 0.2, 0.05] and table["recall"].is_monotonic_increasing and table["precision"].is_monotonic_decreasing, "lowering the threshold the recall rises and the precision drops", "recall_score and precision_score with (p_mal >= threshold)")
    check(table.loc[0.05, "recall"] >= 0.98 and table.loc[0.05, "precision"] < table.loc[0.5, "precision"] - 0.05, "with threshold 0.05 the recall reaches 0.98 or more, at the expense of precision (over 5 points less)", "compare the rows of the table")
    check(0.02 < screening_threshold < 0.4 and recall_score(y_te, (p_mal >= screening_threshold).astype(int)) >= 0.98, "the screening threshold is low and gives a recall of at least 0.98", "thresholds[recalls >= 0.98].max()")
    check(acc_without_scaler < test_acc and iter_without >= 1000, "without scaler the accuracy is lower and the solver hits the limit of 1000 iterations without converging", "without.score(X_te, y_te), without.n_iter_[0]")
done()

---
The commented solution is in `solutions/B_with_sklearn/R4_warmup_the_first_logistic_and_the_threshold.ipynb`. Open it after you pass the check, or after you have really tried.